<img src='./img/cng_thrivegeo_banner.png' class="center" width=95%>


# 🚀 Discover DuckDB

## Introduction

In *Discover Parquet*, you found the buildings of Snowbird by hand: you went to the right folder, picked the file from an index, read its footer, chose the row groups and read three columns. That showed you how skipping works. But for an everyday question, such as *which restaurants are there in the centre of Salt Lake City?*, you don't want to do all of this yourself.

[DuckDB](https://duckdb.org/) is a database engine that answers questions written in **SQL**, a widely used language for asking questions of tables. There is no server to set up: DuckDB runs inside your Python session, and it queries files where they are stored, including Parquet files in the cloud.

DuckDB reads the same folder names, footers and statistics that you read by hand, and uses them to skip data:

* 🗂️ **Folders**: with a path such as `theme=places/type=place`, it opens only the files that can hold your answer
* 🧱 **Row groups**: it checks the statistics in each footer, such as those of the `bbox` columns, and skips the blocks outside your area
* 📊 **Columns**: it reads only the columns your query uses

With the `spatial` extension, DuckDB also understands geometries, so it works with GeoParquet.

Here, you:

1. 🔎 **Write** your first SQL queries on Overture Maps
2. 📈 **Watch** DuckDB skip data, and measure how much it downloads
3. 🍽️ **Find** the restaurants in the centre of Salt Lake City, and map them
4. 🛌 **Find** somewhere to sleep near Snowbird, and save the result as a GeoParquet file

**Prerequisites:** basic Python and *Discover Parquet*. You don't need to know SQL: every query is explained.

### By the end of this tutorial, you will be able to...

* 🦆 **Set up** DuckDB with the `spatial` and `httpfs` extensions to query data in the cloud
* 🔎 **Write** SQL queries with `SELECT`, `FROM`, `WHERE` and `GROUP BY` on Parquet files
* 📈 **Explain** how DuckDB skips folders, row groups and columns, and measure it with `EXPLAIN ANALYZE`
* 🌍 **Turn** a query result into a GeoDataFrame and map it
* 💾 **Save** a query result as a GeoParquet file with `COPY`

### Notebook Outline

* [Example 1️⃣: Your first query](#example1)
* [Exercise 1️⃣ : Restaurants in City Center](#example2)
* [Example 2️⃣: Watch DuckDB skip](#example3)
* [Example 3️⃣: Restaurants in Salt Lake City](#example4)
* [Exercise 1️⃣: Somewhere to sleep near Snowbird](#example5)
* [Takeaways](#example6)

<hr>

**Import libraries**

In [ ]:
import duckdb
import geopandas as gpd
from pathlib import Path

**Load helper function**

In [ ]:
def show_cost(query):
    """Run a query with EXPLAIN ANALYZE and print how much was downloaded and how long it took."""
    report = con.sql("EXPLAIN ANALYZE " + query).fetchone()[1]
    for line in report.splitlines():
        if any(key in line for key in ("in:", "#GET", "Total Time", "Total Files Read")):
            print(line.strip("│ "))

<hr>

<a id="example1"></a> 

## Example 1️⃣: Your first DuckDB query

In this example, you set up DuckDB and learn to read and write SQL, with one question in mind: **what is there in the centre of Salt Lake City?**

1. 🦆 **Start** DuckDB
2. 🔌 **Find** the latest Overture Maps release
3. 🧬 **Look** at the columns
4. 🔎 **Select** a few columns
5. 🎯 **Filter** and count

### 🦆 Step 1: Start DuckDB

`duckdb.connect()` starts a DuckDB **session** inside your Python process. There is no server to install or log in to. You send your SQL queries through the connection `con`.

DuckDB keeps its core small and adds features through **extensions**. You need two:


| Extension | What it adds |
|---|---|
| `spatial` | geometry types and spatial functions |
| `httpfs` | reading files from the web and from cloud storage such as S3 |


`INSTALL` downloads an extension, which is only needed once per computer. `LOAD` switches it on for this session. Overture Maps is stored in the AWS region `us-west-2`, so you also tell DuckDB where to look:

In [ ]:
# Initialise our session
con = duckdb.connect()

con.sql("INSTALL spatial; LOAD spatial;")
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("SET s3_region = 'us-west-2';")

### 🔌 Step 2: Find the latest release

In *Discover Parquet*, you asked Overture's STAC catalog for the latest release with `pystac`. DuckDB can do the same in SQL, because it reads a JSON file from the web **as if it were a table**. The `latest` field of the catalog becomes a column:

In [ ]:
con.sql("SELECT latest FROM 'https://stac.overturemaps.org/catalog.json'")

This is your first SQL query. Read it almost like an English sentence: *select the `latest` column from this file*. DuckDB shows the result as a small table.

To use the value in Python, `.fetchone()` fetches the first row, and `[0]` takes its first value. With it, you build the path of the places files, which follows the folder layout you browsed in *Discover Parquet*:

In [ ]:
latest_release = con.sql("SELECT latest FROM 'https://stac.overturemaps.org/catalog.json'").fetchone()[0]

release_path = f"s3://overturemaps-us-west-2/release/{latest_release}"
places_path = f"{release_path}/theme=places/type=place/*.parquet"

print("Latest release:", latest_release)
print("Places:        ", places_path)

<div class="alert alert-block alert-info">

**Note**

Same data, different tool

These are the same GeoParquet files you browsed with `s3fs`. Only the tool has changed. From here on, you ask your questions in SQL, and DuckDB decides which files, row groups and columns it needs to read.
</div>

### 🧬 Step 3: Look at the columns

`read_parquet()` reads Parquet files. The `*` in the path stands for "all files in this folder", so DuckDB treats **all places files as one big table**. `DESCRIBE` lists the columns of that table and their types. It only reads a footer, not the data:

In [ ]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{places_path}')").df()[["column_name", "column_type"]]

<div class="alert alert-block alert-info">

**Note**

Python variables inside SQL

The queries are Python **f-strings**: the text between `{ }` is replaced by the value of a Python variable before the query goes to DuckDB. So `{places_path}` becomes the full S3 path. Inside SQL, text values such as paths go between single quotes: `'...'`.
</div>

Look at the last two rows: `theme` and `type` are not stored in the files. DuckDB made them from the `key=value` folder names.

You will use four columns:

<center>

| Column | What it holds | How you use it |
|---|---|---|
| `names` | the names of a place | `names.primary`: the main name |
| `basic_category` | the kind of place | e.g. `restaurant` |
| `bbox` | the bounding box, as in *Discover Parquet* | `bbox.xmin`, `bbox.ymin`, … |
| `geometry` | the location, as a point | DuckDB decodes the stored WKB into its own geometry type |

</center>

`names` and `bbox` are **nested** columns (`STRUCT`). In SQL, you reach the part inside with a dot, e.g. `names.primary`.

### 🔎 Step 4: Select a few columns

Most SQL queries follow the same pattern:

<center>

| Part | Question it answers | Example |
|---|---|---|
| `SELECT` | Which **columns** do you want? | `SELECT names.primary, basic_category` |
| `FROM` | From which **files** or table? | `FROM read_parquet('...')` |
| `WHERE` | Which **rows**? | `WHERE basic_category = 'restaurant'` |
| `LIMIT` | How many rows at most? | `LIMIT 10` |

</center>
Start with the first ten places:

In [ ]:
con.sql(f"""
    SELECT
        names.primary  AS name,        -- AS gives the column a shorter name
        basic_category AS category
    FROM read_parquet('{places_path}')
    LIMIT 10
""").df()

`.df()` turns the result into a pandas DataFrame. Text after `--` is a **comment**: DuckDB ignores it.

These ten places come from all over the world. To get to Salt Lake City, you need a filter.

### 🎯 Step 5: Filter and count

First, how many places are there in total? `count(*)` counts rows:

In [ ]:
all_places = con.sql(f"SELECT count(*) FROM read_parquet('{places_path}')").fetchone()[0]

print(f"{all_places:,} places in the world")

Two things are new compared with the Parquet notebook:

* `read_parquet()` reads Parquet files. It accepts a single file or a pattern with `*`, so it treats **all matching files as one big table**.
* With `hive_partitioning=1`, DuckDB turns the `key=value` folder names into columns. Look at the last rows: `theme` and `type` are now columns, even though they are not stored inside the files.

These columns are the most important ones for places:

<center>

| Column | What it holds | How to reach the part you need |
|---|---|---|
| `names` | the names of a place, in several languages | `names.primary`: the main name |
| `categories` | what kind of place it is | `categories.primary`: the main category, e.g. `restaurant` |
| `bbox` | the bounding box, as in the Parquet notebook | `bbox.xmin`, `bbox.ymin`, … |
| `geometry` | the location, as a point | in WKB |

</center>

`names`, `categories` and `bbox` are **nested** columns (`STRUCT`). In SQL, you reach the part inside with a dot, e.g. `names.primary`.

Now keep only the places in a few blocks in the centre of Salt Lake City, the city whose land use you mapped in *Discover Parquet*. `WHERE` keeps the rows that pass a test. As in *Discover Parquet*, you test the `bbox` columns:

In [ ]:
%%time
downtown_bbox = (-111.888199, 40.756360, -111.879530, 40.762731)   # west, south, east, north
west, south, east, north = downtown_bbox

con.sql(f"""
    SELECT count(*) AS places
    FROM read_parquet('{places_path}')
    WHERE bbox.xmin BETWEEN {west} AND {east}
      AND bbox.ymin BETWEEN {south} AND {north}
""").df()

<div class="alert alert-block alert-info">

**Note**

Why only `xmin` and `ymin`?

Places are **points**, so their bounding box has no size: `xmin` equals `xmax`, and `ymin` equals `ymax`. Testing one corner is enough.
</div>

What kinds of places are they? You need the same filter again, so store it in a Python variable. `GROUP BY` puts the rows with the same category together, `count(*)` counts each group, and `ORDER BY … DESC` sorts from the largest to the smallest:

In [ ]:
downtown_filter = f"bbox.xmin BETWEEN {west} AND {east} AND bbox.ymin BETWEEN {south} AND {north}"

con.sql(f"""
    SELECT
        basic_category AS category,
        count(*)       AS places
    FROM read_parquet('{places_path}')
    WHERE {downtown_filter}
    GROUP BY category
    ORDER BY places DESC
    LIMIT 10
""").df()

<div class="alert alert-block alert-success">

<b>Reflection</b>

- ➡️ 1. Look at the category names. How are they written? Why does this matter when you filter with `WHERE`?
- ➡️ 2. DuckDB found the places of a few city blocks among all the places in the world within seconds. How do you think it did that?
</div>

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. Look at the category names. How are they written? Why does this matter when you filter with `WHERE`?<br>
    ✏️ : In lower case, with underscores instead of spaces. In `WHERE`, the text must match exactly: `'Restaurant'` would find nothing.
- ➡️ 2. DuckDB found the places of a few city blocks among all the places in the world within seconds. How do you think it did that?<br>
    ✏️ : It did not read all the places. It used what you saw in *Discover Parquet*: the path limits it to the places folder, the `bbox` statistics in the footers tell it which row groups can hold your area, and it reads only the columns the query names. In the next example, you measure this.

</div>

<hr>

<a id="example2"></a> 

## Exercise 1️⃣ : Restaurants in City Center 🍽️

<div class="alert alert-block alert-warning">

**📝 Instructions**

- ➡️ How many **restaurants** are there in the centre of Salt Lake City? Write a query that counts them, and compare your result with the table above.

**Hint**

- Start from the count query in Step 5. `AND` adds a second test to `WHERE`: `AND basic_category = 'restaurant'`.
</div>

In [ ]:
con.sql(f"""
    SELECT count(*) AS restaurants
    FROM read_parquet('{places_path}')
    WHERE basic_category = 'restaurant'
    AND {downtown_filter}
""").df()

<hr>

<a id="example3"></a> 

## Example 2️⃣: Watch DuckDB skip

In Example 1, DuckDB found the places of a few city blocks among all the places in the world. Now you measure how. You ask a question, change **one thing**, and compare how much DuckDB downloads:

1. 📈 **Measure** a query with `EXPLAIN ANALYZE`
2. 🗂️ **Files**: find the file that holds your places
3. 🧱 **Row groups**: compare two ways of filtering by location
4. 📊 **Columns**: compare one column with all columns

### 📈 Step 1: Measure a query

DuckDB keeps what it has downloaded in memory, so a second look at the same data costs nothing. That is useful in daily work, but it gets in the way of measuring: a repeated query would report almost no download. Switch this memory off for the rest of this session:

In [ ]:
con.sql("SET enable_external_file_cache = false;")

If you put `EXPLAIN ANALYZE` in front of a query, DuckDB runs the query and returns a report of how it did the work, in place of the result. Try it on the count from Example 1. `.fetchone()[1]` takes the report out of the result:

In [ ]:
count_query = f"""
    SELECT count(*) AS places
    FROM read_parquet('{places_path}')
    WHERE {downtown_filter}
"""

report = con.sql("EXPLAIN ANALYZE " + count_query).fetchone()[1]
print(report)

The report is long, but you only need three parts of it:

| Part | Where | What it tells you |
|---|---|---|
| **HTTPFS HTTP Stats** | at the top | `in:` how much was **downloaded**, `#GET:` how many requests were sent |
| **Total Time** | near the top | how long the query took |
| **TABLE_SCAN** | the box at the bottom | the `Filters` that DuckDB applied while reading, and `Total Files Read` |

Look at the `Filters` in the bottom box: DuckDB applied your `bbox` test **while reading** the files. That is where the skipping happens.

### 🗂️ Step 2: Find the file that holds your places

The report says how many files DuckDB opened. But how many of them hold places of your area? With `filename=true`, DuckDB adds a column with the name of the file each row comes from:

In [ ]:
downtown_files = con.sql(f"""
    SELECT filename, count(*) AS places
    FROM read_parquet('{places_path}', filename=true)
    WHERE {downtown_filter}
    GROUP BY filename
""").df()

downtown_files

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. How many files hold places of your area? How many files did DuckDB open in Step 1 (`Total Files Read`)?<br>
    ✏️ : One file holds all of them. DuckDB opened every places file, but none of the other themes: the path keeps it inside the places folder.
- ➡️ 2. In *Discover Parquet*, you used an index to pick the right file. DuckDB has no index. How does it find your places anyway?<br>
    ✏️ : It reads the **footer** of every places file and compares the `bbox` statistics of each row group with your box. In all files but one, no row group can match, so DuckDB reads nothing more than their footers. Footers are small, so this is cheap.
</div>

### 🧱 Step 3: Compare two ways of filtering by location

A place says where it is in two columns: `bbox` and `geometry`. For points, `ST_X()` and `ST_Y()` give the longitude and latitude of the geometry. So you can ask for the places in your box in two ways:

* 📦 **with `bbox`**: `bbox.xmin BETWEEN …`
* 📍 **with the geometry**: `ST_X(geometry) BETWEEN …`

To keep the comparison short, use only the file that holds your places:

In [ ]:
downtown_file = downtown_files["filename"][0]

bbox_query = f"""
    SELECT count(*) FROM read_parquet('{downtown_file}')
    WHERE {downtown_filter}
"""
geometry_query = f"""
    SELECT count(*) FROM read_parquet('{downtown_file}')
    WHERE ST_X(geometry) BETWEEN {west} AND {east}
      AND ST_Y(geometry) BETWEEN {south} AND {north}
"""

<div class="alert alert-block alert-success">

**Before you run the next cell - Answer**

- ✏️ : The `bbox` query downloads far less and finishes sooner. For the `bbox` columns, the footer stores the smallest and largest value of each row group, so DuckDB skips every row group that cannot match. The geometry is stored as WKB, just bytes, so its statistics say nothing about location. For the second query, DuckDB has to download **every geometry** in the file and test each point.

    This is why Overture adds the `bbox` column, and why you could not look for Vienna in EuroCrops, which has none.
</div>

In [ ]:
print("📦 With bbox:")
show_cost(bbox_query)

print("\n📍 With the geometry:")
show_cost(geometry_query)

### 📊 Step 4: Compare one column with all columns

The last level is the simplest. Ask for the same rows, once with only the names, and once with **all** columns (`*`):

In [ ]:
names_query = f"SELECT names.primary FROM read_parquet('{downtown_file}') WHERE {downtown_filter}"
all_query = f"SELECT * FROM read_parquet('{downtown_file}') WHERE {downtown_filter}"

print("One column:")
show_cost(names_query)

print("\nAll columns:")
show_cost(all_query)

Naming your columns downloads less, because every column is stored on its own inside a row group. `SELECT *` is handy for a first look, but name your columns once you know what you need.

<div class="alert alert-block alert-info">

**Note**

💡 Remember...

DuckDB skips data at the levels you explored by hand in *Discover Parquet*, and your query controls each one:

| Level | What DuckDB uses | What you write |
|---|---|---|
| 🗂️ **Folders** | the `key=value` folder names | a narrow path, e.g. `theme=places/type=place` |
| 🧱 **Row groups** | the statistics in the footers | a filter on columns with useful statistics, such as `bbox`, not on the geometry |
| 📊 **Columns** | the column-by-column storage | only the columns you need after `SELECT` |

DuckDB has no index of files. It opens every footer in the folder, then skips the files in which no row group can match.
</div>

<hr>

<a id="example4"></a> 

## Example 3️⃣: Restaurants in Salt Lake City

Time to answer the question from the introduction: *which restaurants are there in the centre of Salt Lake City?* You go from the question to a map and saving your data in four steps:

1. 🔎 **Write** the query
2. 📋 **Run** it and look at the result
3. 🗺️ **Put** the restaurants on a map
4. 💾 **Save** your result

### 🔎 Step 1: Write the query

You already have all the pieces: the path of the places files (`places_path`), the location filter (`downtown_filter`) and the category column (`basic_category`). `AND` joins the two filters, so a place must pass both:

In [ ]:
restaurants_query = f"""
    SELECT
        id,
        names.primary            AS name,
        ST_AsWKB(geometry)::BLOB AS geometry_wkb
    FROM read_parquet('{places_path}')
    WHERE basic_category = 'restaurant'
      AND {downtown_filter}
"""

One line is new: `ST_AsWKB(geometry)::BLOB`. DuckDB reads the geometries into its own geometry type, which pandas does not understand. `ST_AsWKB()` turns them back into standard **WKB**, and `::BLOB` hands them over as plain bytes.

### 📋 Step 2: Run it and look at the result

`con.sql()` sends the query to DuckDB, and `.df()` returns the result as a pandas DataFrame:

In [ ]:
%%time
restaurants = con.sql(restaurants_query).df()
print(len(restaurants), "restaurants")
restaurants.head()

<div class="alert alert-block alert-success">

**Reflection and answers** 

- ➡️ 1. What do you see in the `geometry_wkb` column? Where have you met this before?<br>
    ✏️ : A row of bytes. It is the **WKB** encoding from *Discover Parquet*: the location of each restaurant, stored in binary. To draw the restaurants on a map, you have to decode it.
- ➡️ 2. The query has two filters: one on the category and one on `bbox`. Which of them lets DuckDB skip row groups? Why does the other one not help much?<br>
    ✏️ : The `bbox` filter. Overture sorts the places by location, so each row group covers a compact area, and its `bbox` statistics can rule it out. Restaurants, however, are found everywhere: almost every row group holds *some* restaurant, so the statistics of `basic_category` rule out very little. DuckDB still applies the category filter, but only inside the row groups that the `bbox` filter let through.
</div>

### 🗺️ Step 3: Put the restaurants on a map

As in *Discover Parquet*, `gpd.GeoSeries.from_wkb()` decodes the WKB into shapes. One small detail: DuckDB delivers the bytes as a `bytearray`, while `from_wkb()` expects `bytes`, so you convert them first with `.apply(bytes)`:

In [ ]:
restaurants_map = gpd.GeoDataFrame(
    restaurants.drop(columns="geometry_wkb"),
    geometry=gpd.GeoSeries.from_wkb(restaurants["geometry_wkb"].apply(bytes), crs="EPSG:4326"),
)

restaurants_map.explore(tooltip="name", marker_kwds={"radius": 6})

From a question to a map in three steps:

1. 🔎 **Query** with DuckDB: a narrow path, a `bbox` filter and only the columns you need
2. 📦 **Hand over** the geometries as WKB with `ST_AsWKB()`, and fetch the result with `.df()`
3. 🗺️ **Decode** them with `gpd.GeoSeries.from_wkb()`, and map them with `.explore()`

You use exactly these steps in the exercise.

### 💾 Step 4: Save your result

So far, every result has lived only in memory: close the notebook and it is gone. To keep a result, wrap the query in `COPY ( … ) TO 'file' (FORMAT PARQUET)`. DuckDB then writes the rows to a file, straight from the cloud to your disk.

One detail matters: select the `geometry` column **as it is**, without `ST_AsWKB()`. DuckDB then writes the `geo` entry into the footer, and that entry is what makes the file a **GeoParquet** file. Run the cell, then read the file back with `geopandas`:

In [ ]:
Path("results").mkdir(exist_ok=True)
restaurants_path = "results/slc_restaurants.parquet"

con.sql(f"""
    COPY (
        SELECT
            id,
            names.primary AS name,
            geometry
        FROM read_parquet('{places_path}')
        WHERE basic_category = 'restaurant'
          AND {downtown_filter}
    ) TO '{restaurants_path}' (FORMAT PARQUET);
""")

saved = gpd.read_parquet(restaurants_path)

print(len(saved), "restaurants, CRS:", saved.crs)
saved.head()

<hr>

<a id="example5"></a> 

## Exercise 2️⃣ : Somewhere to sleep near Snowbird

In *Discover Parquet*, you loaded the buildings of Snowbird. Now imagine you are planning a ski trip there: where could you stay? Use the steps from Example 3 to find the places to sleep around Snowbird 🛌, and put them on a map.

This is the same area as before:

In [ ]:
snowbird_bbox = (-111.724091, 40.517975, -111.493378, 40.640530)   # west, south, east, north

<div class="alert alert-block alert-warning">

**Instructions**

- ➡️ 1. Write a location filter for `snowbird_bbox`, like `downtown_filter` in Example 1. Call it `snowbird_filter`.<br>
- ➡️ 2. Query the id, name, category and geometry of all places to sleep in the area. Call the result `sleep_places`. How many did you find?<br>
- ➡️ 3. Put them on a map, coloured by category.

**Hints**

- Places are points, so `bbox.xmin` and `bbox.ymin` are enough.
- For task 2, pick the categories from the overview in the checkpoint cell. To filter on several categories at once, use `IN`: `WHERE basic_category IN ('hotel', 'resort')`.
- The query and the map work as in Example 3. In `.explore()`, `column="category"` colours the points, and `tooltip=["name", "category"]` shows both columns.
</div>

#### ➡️ Write the location filter

In [ ]:
west, south, east, north = snowbird_bbox

snowbird_filter = f"bbox.xmin BETWEEN {west} AND {east} AND bbox.ymin BETWEEN {south} AND {north}"

In [ ]:
con.sql(f"""
    SELECT
        basic_category AS category,
        count(*)           AS places
    FROM read_parquet('{places_path}')
    WHERE {snowbird_filter}
    GROUP BY category
    ORDER BY places DESC
    LIMIT 20
""").df()

Among the most common categories, several are places to sleep, e.g. [`hotel`, `resort`, `lodging`, …].

#### ➡️ Query the places to sleep

In [ ]:
sleep_categories = "('hotel', 'resort', 'lodging', 'private_lodging')"

sleep_query = f"""
    SELECT
        id,
        names.primary            AS name,
        basic_category           AS category,
        ST_AsWKB(geometry)::BLOB AS geometry_wkb
    FROM read_parquet('{places_path}')
    WHERE basic_category IN {sleep_categories}
      AND {snowbird_filter}
"""

sleep_places = con.sql(sleep_query).df()

print(len(sleep_places), "places to sleep")
sleep_places.head()

#### ➡️ 3. Put the place to sleep on a map

In [ ]:
sleep_map = gpd.GeoDataFrame(
    sleep_places.drop(columns="geometry_wkb"),
    geometry=gpd.GeoSeries.from_wkb(sleep_places["geometry_wkb"].apply(bytes), crs="EPSG:4326"),
)

sleep_map.explore(column="category", tooltip=["name", "category"], marker_kwds={"radius": 6})

After we are done with our extraction, and stored our results, it is a good practice to close our DuckDB Session

In [ ]:
con.close()

<hr>

<a id="example6"></a> 

## Takeaways 

You started with a link to the Overture catalog and ended with a map of places to sleep near Snowbird, saved as a file on your own disk. You never downloaded the dataset. DuckDB fetched only the pieces your questions needed.

**What you learned**

* 🦆 **A database inside your Python session:** `duckdb.connect()` starts DuckDB without a server. The `spatial` and `httpfs` extensions add geometries and cloud storage.
* 🔎 **SQL follows your question:** `SELECT` picks the columns, `FROM` names the files, `WHERE` keeps the rows you need, and `GROUP BY` counts them by group. With `read_parquet('…/*.parquet')`, a whole folder of files behaves like one table.
* 📈 **DuckDB skips for you:** it uses the folder names, the footer statistics and the column-by-column storage that you explored by hand in *Discover Parquet*. `EXPLAIN ANALYZE` shows how much a query downloads.
* 🌍 **From a query to a map:** `ST_AsWKB()` hands the geometries over as WKB, `gpd.GeoSeries.from_wkb()` decodes them, and `.explore()` draws them.
* 💾 **Keep your result:** `COPY ( … ) TO` writes a query result to a file. If you keep the `geometry` column as it is, the file is GeoParquet.

**Things to keep in mind**

* 📦 **How you filter matters.** A filter on the `bbox` columns lets DuckDB skip row groups. A filter on the geometry makes it read every geometry first. Two queries can return the same rows and download very different amounts.
* 📊 **Name your columns.** `SELECT *` on a cloud dataset downloads every column, including the heavy ones.
* 🔀 **Skipping depends on how the data was written.** It works well for Overture Maps, which is sorted by location and has a `bbox` column. Not every dataset is written this way.
* 🧠 **DuckDB remembers what it has downloaded.** You switched this memory off to measure. In your own work, leave it on: a repeated query is then much faster.
* 🔍 **A result is only as good as its data.** The `resort` category also held ski lifts. Look at your result on a map before you trust it.

**Next steps**

This is the last part of the workshop. Look back at the path you took:

* 🔭 **STAC** helped you *find* data.
* 🖼️ **COG** and **Zarr** let you *read only the part you need* from images and data cubes.
* 📋 **Parquet** showed you *how* tables are organised so that a reader can skip.
* ⚙️ **Dask** let you *compute* on more data than fits into memory.
* 🦆 **DuckDB** *answered questions* about tables in the cloud with a single query.

Now try it on your own area: change the bounding box, pick another Overture theme (`buildings`, `transportation` or `base`), and save the result as GeoParquet to open in QGIS.

📚 *Further reading:* [DuckDB spatial extension](https://duckdb.org/docs/stable/core_extensions/spatial/overview) · [DuckDB and S3](https://duckdb.org/docs/stable/core_extensions/httpfs/overview) · [Overture Maps with DuckDB](https://docs.overturemaps.org/getting-data/duckdb/) · [GeoParquet specification](https://geoparquet.org/) · [Cloud-Optimized Geospatial Formats Guide](https://guide.cloudnativegeo.org/)

<hr>

&copy; 2026 thriveGEO GmbH